# RK1 eval overnight: base vs S1 (plant + general), judge, checkpoint rule, test

Без переобучения: берёт адаптер с Drive `MyDrive/rk1_houseplant/s1_plant_qlora`.

1. Runtime → T4 GPU.
2. Secrets (ключ слева) → `OPENROUTER_API_KEY`, включить Notebook access.
3. Runtime → Run all. Вкладку не закрывать.

Всё пишется сразу в `MyDrive/rk1_houseplant/eval_s1/` построчно. Если runtime отвалился — просто Run all ещё раз: готовое пропускается.

Для быстрой проверки поставьте `SMOKE = True` (3 примера на набор, отдельная папка).

In [ ]:
from google.colab import drive, userdata
import os

SMOKE = False

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rk1_houseplant'
ADAPTER = f'{DRIVE}/s1_plant_qlora'
OUT = f'{DRIVE}/eval_smoke' if SMOKE else f'{DRIVE}/eval_s1'
assert os.path.exists(f'{ADAPTER}/adapter_model.safetensors'), f'adapter not found in {ADAPTER}'
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.makedirs(OUT, exist_ok=True)
print('adapter:', sorted(os.listdir(ADAPTER)))
print('out:', OUT)

In [ ]:
%cd /content
!git clone https://github.com/Schaft-s/LLM || (cd LLM && git pull)
%cd /content/LLM
!bash scripts/colab_install.sh
!git rev-parse HEAD > "{OUT}/git_commit.txt"
!mkdir -p "{OUT}/configs" && cp configs/*.yaml "{OUT}/configs/"
!nvidia-smi --query-gpu=name,memory.total --format=csv > "{OUT}/gpu.txt"

In [ ]:
LIMIT = '--limit 3' if SMOKE else ''
!python scripts/run_eval_pipeline.py --out-dir "{OUT}" --adapter-root "{ADAPTER}" {LIMIT} 2>&1 | tee -a "{OUT}/run.log"

In [ ]:
import glob
from IPython.display import Image, Markdown, display

display(Markdown(open(f'{OUT}/summary.md').read()))
for p in sorted(glob.glob(f'{OUT}/plots/*.png')):
    print(p)
    display(Image(p))
drive.flush_and_unmount()
print('Drive flushed')